In [7]:
from collections import Counter

# =========================================================
#                     NAIVE BAYES
# =========================================================

def train_naive_bayes(X, y):
    total_lignes = len(y)
    compte_classes = Counter(y)

    # 1. Probabilités de base : P(OUI) et P(NON) dans tout le dataset
    prob_prior = {c: count/total_lignes for c, count in compte_classes.items()}

    # 2. Probabilités des détails : P(Caractéristique | Classe)
    model = {}
    for c in compte_classes:
        # On ne garde que les lignes de la classe actuelle
        lignes_c = [X[i] for i in range(len(X)) if y[i] == c]
        model[c] = []

        # On parcourt chaque colonne pour voir les habitudes
        for col in range(len(X[0])):
            frequences = Counter([ligne[col] for ligne in lignes_c])
            # On stocke le pourcentage d'apparition
            model[c].append({val: count/len(lignes_c) for val, count in frequences.items()})

    return prob_prior, model

def predict_naive_bayes(instance, prob_prior, model):
    scores = {}
    for c in prob_prior:
        # On part de la probabilité de base de la classe
        prob = prob_prior[c]
        # On multiplie par la probabilité de chaque caractéristique observée
        for i, val in enumerate(instance):
            # Si on n'a jamais vu cette valeur, on met une proba minuscule (0.0001) pour ne pas annuler tout
            prob *= model[c][i].get(val, 0.0001)
        scores[c] = prob

    # On choisit la classe qui obtient le score le plus élevé
    return max(scores, key=scores.get)

# =========================================================
#                     TEST DU MODÈLE
# =========================================================

# Données : [Météo, Température]
X_train = [["Soleil", "Chaud"], ["Soleil", "Chaud"], ["Pluie", "Froid"], ["Pluie", "Froid"]]
y_train = ["OUI", "OUI", "NON", "NON"]

print("--- TEST NAIVE BAYES ---")
prior, mon_modele = train_naive_bayes(X_train, y_train)
nouveau_cas = ["Soleil", "Froid"]
resultat = predict_naive_bayes(nouveau_cas, prior, mon_modele)
print(f"Situation : {nouveau_cas} => Prédiction : {resultat}")

--- TEST NAIVE BAYES ---
Situation : ['Pluie', 'chaude'] => Prédiction : NON


In [8]:
import math

# =========================================================
#                         DBSCAN
# =========================================================
  # on calcule la distance entre deux points.
def distance_euclidienne(p1, p2):
    return math.sqrt(sum((p1[i] - p2[i])**2 for i in range(len(p1))))
  # on essaye de trouve tous les points qui sont dans la zone
def get_voisins(X, point_idx, eps):
    return [i for i, p in enumerate(X) if distance_euclidienne(X[point_idx], p) <= eps]

def dbscan_from_scratch(X, eps, min_pts):

    #X : Liste de points
    #eps : Distance maximum pour être voisins
    #min_pts : Nombre minimum de points pour dire que c'est un 'groupe dense'

    # labels : 0 = Non traité, -1 = Anomalie, 1,2... = Identifiant du groupe
    labels = [0] * len(X)
    cluster_id = 0

    for i in range(len(X)):
        if labels[i] != 0: continue # Si déjà analysé, on passe au suivant

        voisins = get_voisins(X, i, eps)

        if len(voisins) < min_pts:
            labels[i] = -1 # Pas assez de monde autour : c'est peut-être du bruit
        else:
            cluster_id += 1 # On a trouvé le cœur d'un nouveau groupe !
            labels[i] = cluster_id

            # ÉTAPE : On essaie d'étendre le groupe aux voisins des voisins
            a_analyser = voisins
            for v_idx in a_analyser:
                if labels[v_idx] == -1: labels[v_idx] = cluster_id # Une anomalie qui touche un groupe devient un bord
                if labels[v_idx] != 0: continue # Déjà dans un groupe

                labels[v_idx] = cluster_id # On l'ajoute au groupe
                nouveaux_voisins = get_voisins(X, v_idx, eps)
                if len(nouveaux_voisins) >= min_pts:
                    # Si le voisin est aussi un 'cœur', on continue de s'étendre par lui
                    a_analyser.extend(nouveaux_voisins)

    return labels

# =========================================================
#                      TEST DU MODÈLE
# =========================================================

# Problématique : On a deux paquets de points et un point tout seul à l'écart [10, 10]
X_points = [
    [1, 1], [1.1, 1], [1, 0.9],   # Groupe 1 (proches)
    [5, 5], [5.1, 5.2], [4.9, 5], # Groupe 2 (proches)
    [10, 10]                      # Anomalie (très loin)
]

print("\n--- TEST DBSCAN ---")
# eps=1.5 : on cherche à 1.5 de distance / min_pts=2 : il faut être au moins 2
resultats = dbscan_from_scratch(X_points, eps=1.5, min_pts=2)
print(f"Résultats ( -1 veut dire ANOMALIE ) : {resultats}")


--- TEST DBSCAN ---
Résultats ( -1 veut dire ANOMALIE ) : [1, 1, 1, 2, 2, 2, -1]
